# Explainability (SHAP) & Population Stability Index (PSI) Drift - Project 2
> **Production Alignment Note:** Loads the registered `Production` model (`loan_default_production_model`) with the canonical 22-feature ECOA/Reg-B compliant matrix (`MaritalStatus` excluded) and runs `DataDriftMonitor` (`src/validation/drift_check.py`).

In [ ]:
import sys
from pathlib import Path
import mlflow
import mlflow.xgboost
import numpy as np
import pandas as pd
import shap
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.inference.main import EXPECTED_LOAN_FEATURES
from src.validation.drift_check import DataDriftMonitor

DB_PATH = PROJECT_ROOT.parent / "mlflow_db" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
client = MlflowClient()
versions = client.search_model_versions("name='loan_default_production_model'")
prod_versions = [v for v in versions if v.current_stage == "Production"]
if not prod_versions:
    raise RuntimeError("No Production version found for loan_default_production_model")
prod_mv = max(prod_versions, key=lambda v: int(v.version))
model = mlflow.xgboost.load_model(
    f"models:/loan_default_production_model/{prod_mv.version}"
)
test_df = pd.read_parquet(PROJECT_ROOT / "data/processed/test.parquet")

In [ ]:
drop_cols = [
    c
    for c in ("LoanID", "Default", "MaritalStatus", "is_young")
    if c in test_df.columns
]
X_test = pd.get_dummies(test_df.drop(columns=drop_cols), drop_first=True)
X_test = X_test.reindex(columns=EXPECTED_LOAN_FEATURES, fill_value=0)

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(X_test.iloc[:500])
top_shap = pd.Series(
    np.abs(shap_vals).mean(axis=0), index=EXPECTED_LOAN_FEATURES
).sort_values(ascending=False)
print("Top 5 SHAP Features:\n", top_shap.head(5))

In [ ]:
monitor = DataDriftMonitor(
    reference_path=PROJECT_ROOT / "data/processed/train.parquet",
    artifacts_dir=PROJECT_ROOT / "artifacts",
)
summary = monitor.compute_drift_summary(
    test_df, source_label="holdout_evaluation_split"
)
print("Holdout Drift Status:", summary["overall_status"])